In [1]:
!nvidia-smi

Thu Jul  9 11:28:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
%%writefile hello.cu
#include <cstdio>

// __global__ = "bu fonksiyon GPU'da çalışir, CPU'dan cagrilir" demek.
// Bu fonksiyonlara "kernel" denir.
__global__ void hello()
{
    // Her thread kendi kimligini yazdirir.
    printf("Merhaba! Block %d, Thread %d\n", blockIdx.x, threadIdx.x);
}

int main()
{
    // <<<2, 4>>> : 2 block, her block'ta 4 thread => toplam 8 thread.
    hello<<<2, 4>>>();

    // GPU asenkron calisir; bitmesini beklemezsek program cikti gormeden biter.
    cudaDeviceSynchronize();
    return 0;
}

Writing hello.cu


In [4]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

Merhaba! Block 0, Thread 0
Merhaba! Block 0, Thread 1
Merhaba! Block 0, Thread 2
Merhaba! Block 0, Thread 3
Merhaba! Block 1, Thread 0
Merhaba! Block 1, Thread 1
Merhaba! Block 1, Thread 2
Merhaba! Block 1, Thread 3


In [5]:
%%writefile vector_add.cu
#include <cstdio>
#include <cstdlib>

// Her thread dizinin TEK BIR elemanini toplar.
__global__ void vectorAdd(const float* a, const float* b, float* c, int n)
{
    // Global thread index: kacinci block'tayim * block boyutu + block icindeki sira
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Thread sayisi n'den fazla olabilir; tasanlari engelle.
    if (i < n)
        c[i] = a[i] + b[i];
}

int main()
{
    const int N = 1 << 20;               // 1.048.576 eleman (2^20)
    size_t bytes = N * sizeof(float);

    // 1) CPU (host) bellegi
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_a[i] = 1.0f; h_b[i] = 2.0f; }

    // 2) GPU (device) bellegi
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // 3) Host -> Device kopyalama
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // 4) Kernel launch: N elemani kapatacak kadar block ac
    int blockSize = 256;
    int gridSize = (N + blockSize - 1) / blockSize;  // yukari yuvarlama: 4096 block
    vectorAdd<<<gridSize, blockSize>>>(d_a, d_b, d_c, N);

    // 5) Device -> Host sonucu geri al (cudaMemcpy ayni zamanda senkronize eder)
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // 6) Dogrulama: her eleman 3.0 olmali
    bool ok = true;
    for (int i = 0; i < N; i++)
        if (h_c[i] != 3.0f) { ok = false; break; }
    printf(ok ? "DOGRU: tum elemanlar 3.0\n" : "HATA!\n");

    // 7) Temizlik
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}

Writing vector_add.cu


In [6]:
!nvcc -arch=sm_75 vector_add.cu -o vector_add && ./vector_add

DOGRU: tum elemanlar 3.0


In [7]:
%%writefile matmul.cu
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>

// Naive GPU matris carpimi: her thread, C'nin TEK BIR hucresini hesaplar.
// C[row][col] = A'nin row. satiri ile B'nin col. sutununun ic carpimi.
__global__ void matmulGPU(const float* A, const float* B, float* C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
            sum += A[row * N + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

// Karsilastirma icin klasik CPU versiyonu (3 ic ice dongu).
void matmulCPU(const float* A, const float* B, float* C, int N)
{
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++)
                sum += A[i * N + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

int main()
{
    const int N = 1024;                  // 1024x1024 matris
    size_t bytes = N * N * sizeof(float);

    float *h_A = (float*)malloc(bytes);
    float *h_B = (float*)malloc(bytes);
    float *h_C = (float*)malloc(bytes);      // GPU sonucu
    float *h_ref = (float*)malloc(bytes);    // CPU sonucu (referans)

    for (int i = 0; i < N * N; i++) {
        h_A[i] = (float)(rand() % 10);
        h_B[i] = (float)(rand() % 10);
    }

    // ---------- CPU olcumu ----------
    auto t0 = std::chrono::high_resolution_clock::now();
    matmulCPU(h_A, h_B, h_ref, N);
    auto t1 = std::chrono::high_resolution_clock::now();
    double cpu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

    // ---------- GPU hazirlik ----------
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // 2D grid: 16x16 = 256 thread'lik block'lar, 64x64 block'luk grid
    dim3 blockDim(16, 16);
    dim3 gridDim((N + 15) / 16, (N + 15) / 16);

    // ---------- cudaEvent ile GPU olcumu ----------
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matmulGPU<<<gridDim, blockDim>>>(d_A, d_B, d_C, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_ms = 0.0f;
    cudaEventElapsedTime(&gpu_ms, start, stop);

    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    // ---------- Dogrulama ----------
    bool ok = true;
    for (int i = 0; i < N * N; i++)
        if (fabs(h_C[i] - h_ref[i]) > 1e-3) { ok = false; break; }

    printf("Dogrulama : %s\n", ok ? "BASARILI" : "HATALI");
    printf("CPU suresi: %.2f ms\n", cpu_ms);
    printf("GPU suresi: %.2f ms\n", gpu_ms);
    printf("Hizlanma  : %.1fx\n", cpu_ms / gpu_ms);

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C); free(h_ref);
    return 0;
}

Writing matmul.cu


In [8]:
!nvcc -arch=sm_75 -O2 matmul.cu -o matmul && ./matmul

Dogrulama : BASARILI
CPU suresi: 3587.18 ms
GPU suresi: 5.24 ms
Hizlanma  : 684.6x
